# Ram's Video Effects — Colab

84 video effects (62 + 22 Vox-style) and voiced videos from a script, no install on your computer. Run the cells from top to bottom (▶).

**Nepali:** Mathi bata tala samma ▶ thichdai janus. 2 number cell ma video wa photo upload garnus, 3 ma effect lekhnus, 5 ma video download hunchha.
Video/photo bina, Nepali voice sanga pura video banauna: 1 chalaunus, ani 6 ra 7 number cell.

In [ ]:
#@title 1. Setup (about 1 minute)
!git clone -q https://github.com/ramkesh430/ram-video-effects /content/ram-video-effects 2>/dev/null || (cd /content/ram-video-effects && git pull -q)
%cd /content/ram-video-effects
!bash install.sh 2>&1 | grep -E 'OK|model|Done'

In [ ]:
#@title 2. Upload your video or photo
from google.colab import files
up = files.upload()
SRC = '/content/' + list(up.keys())[0]
open(SRC, 'wb').write(up[list(up.keys())[0]])
print('Using', SRC)

In [ ]:
#@title 3. Choose effects (edit this list)
#@markdown Every effect needs `start` and `end` in seconds. Vox effects (paper-room, vox-bars, map-pin ...) work here too. All effects: run cell 3b or see EFFECTS.md.
PLAN = [
  {"fx": "backdrop-word", "start": 0, "end": 3, "text": "RAM"},
  {"fx": "glitter-spotlight", "start": 3, "end": 5.5},
  {"fx": "neon-outline", "start": 5.5, "end": 8, "color": "#FF4FA3"},
  {"fx": "red-collage", "start": 8, "end": 10, "word": "FOREVER"},
]
import json; json.dump(PLAN, open('/content/plan.json', 'w'), indent=1)
print('Saved', len(PLAN), 'effects')

In [ ]:
#@title 3b. (optional) List all 84 effects (62 + 22 Vox) and their options
!python vox/vox.py --list

In [ ]:
#@title 4. Preview a few frames first
from IPython.display import Image as Img, display
import glob, os
times = ','.join(str(round((e['start'] + e['end']) / 2, 2)) for e in PLAN)
!python vox/vox.py "$SRC" /content/preview.png --plan /content/plan.json --preview $times 2>&1 | grep png
for p in sorted(glob.glob('/content/preview_*.png')):
    print(os.path.basename(p)); display(Img(p, width=260))

In [ ]:
#@title 5. Render and download
OUT = os.path.splitext(SRC)[0] + '_effects.mp4'
!python vox/vox.py "$SRC" "$OUT" --plan /content/plan.json
from IPython.display import HTML
from base64 import b64encode
display(HTML('<video width=320 controls src="data:video/mp4;base64,' + b64encode(open(OUT, 'rb').read()).decode() + '"></video>'))
files.download(OUT)

## Voiced video from a script (no footage needed)

Pick an example script or write your own: each scene has `say` (what the AI voice says) and `fx` (Vox effects).
Nepali: script chhannus wa aafno lekhnus, format chhannus, ani ▶ thichnus. 60 sec ko video banna 10-15 minute lagchha.

In [ ]:
#@title 6. Choose a script
SCRIPT = "vox/scripts/ram_digital_promo.json" #@param ["vox/scripts/ram_digital_promo.json", "vox/scripts/digital_marketing_ne.json", "vox/scripts/nepal_digital_marketing_doc.json"]
#@markdown Or upload your own script JSON (leave unticked to use the one above):
UPLOAD_OWN = False #@param {type:"boolean"}
if UPLOAD_OWN:
    up = files.upload(); SCRIPT = '/content/' + list(up.keys())[0]
    open(SCRIPT, 'wb').write(up[list(up.keys())[0]])
import json
S = json.load(open(SCRIPT, encoding='utf-8'))
for i, sc in enumerate(S['scenes'], 1):
    print(i, sc['say'][:70], '|', ', '.join(f['fx'] for f in sc['fx']))

In [ ]:
#@title 7. Make the voiced video and download
FORMAT = "reel 9:16" #@param ["reel 9:16", "youtube 16:9", "both"]
import os
name = os.path.splitext(os.path.basename(SCRIPT))[0]
jobs = {"reel 9:16": [("1080x1920", "_reel")], "youtube 16:9": [("1920x1080", "_youtube")]}
jobs["both"] = jobs["reel 9:16"] + jobs["youtube 16:9"]
for size, suf in jobs[FORMAT]:
    out = f'/content/{name}{suf}.mp4'
    !python vox/make_video.py "$SCRIPT" --size $size --out "$out"
    files.download(out)